# Laboratorijska vježba 5: Neuronske mreže u regresiji


# Zadatak 1 - Predviđanje cijena kuća u Bostonu

### a) Učitavanje Boston Housing Price skupa podataka

In [ ]:
from tensorflow.keras.datasets import boston_housing

(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()
print('Podaci uspješno učitani.')

### b) Oblik podataka i broj atributa

In [ ]:
print(f'Oblik trening podataka: {train_data.shape}')
print(f'Oblik testnih podataka: {test_data.shape}')
print(f'Broj atributa (značajki): {train_data.shape[1]}')

### c) Opseg vrijednosti (min i max) za atribute i ciljne vrijednosti

In [ ]:
import numpy as np

print('Opseg vrijednosti atributa (min - max):')
for i in range(train_data.shape[1]):
    print(f' Atribut {i+1}: {np.min(train_data[:, i]):.2f} - {np.max(train_data[:, i]):.2f}')

print(f'\nOpseg ciljnih vrijednosti (cijena): {np.min(train_targets):.2f} - {np.max(train_targets):.2f}')

### d) Skaliranje podataka korištenjem MinMaxScaler-a

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
# Fit samo na trening podacima
scaler.fit(train_data)

train_data_scaled = scaler.transform(train_data)
test_data_scaled = scaler.transform(test_data)

print('Skaliranje završeno.')
print(f'Nova max vrijednost prvog atributa: {np.max(train_data_scaled[:, 0])}')

### e) Pomoćna funkcija `model_mreze()`

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def model_mreze():
    model = Sequential([
        Dense(64, activation='relu', input_shape=(train_data.shape[1],)),
        Dense(64, activation='relu'),
        Dense(1)
    ])

    # f) Kompajliranje modela
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

print('Funkcija za model definisana.')

### g) Treniranje modela na 100 epoha

In [ ]:
import matplotlib.pyplot as plt

model = model_mreze()

history = model.fit(
    train_data_scaled, train_targets,
    epochs=100,
    batch_size=1,
    validation_split=0.1,
    verbose=0 # Smanjujemo ispis radi preglednosti
)

print(f'Završni MSE: {history.history["loss"][-1]:.4f}')
print(f'Završni MAE: {history.history["mae"][-1]:.4f}')

# Grafički prikaz
epochs_range = range(1, 101)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['loss'], label='Trening Loss (MSE)')
plt.plot(epochs_range, history.history['val_loss'], label='Validacijski Loss (MSE)')
plt.title('MSE Gubitak')
plt.xlabel('Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['mae'], label='Trening MAE')
plt.plot(epochs_range, history.history['val_mae'], label='Validacijski MAE')
plt.title('MAE Metrika')
plt.xlabel('Epochs')
plt.legend()

plt.show()

### h) Prikaz usrednjenih vrijednosti (smooth curve) i zanemarivanje prvih 10 uzoraka

In [ ]:
def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

mae_history = history.history['val_mae']
smooth_mae_history = smooth_curve(mae_history[10:])

plt.plot(range(1, len(smooth_mae_history) + 1), smooth_mae_history)
plt.xlabel('Epochs')
plt.ylabel('Validation MAE')
plt.title('Usrednjeni Validacijski MAE (bez prvih 10 epoha)')
plt.show()

### i) Treniranje na 500 epoha i uporedba

In [ ]:
model_500 = model_mreze()

history_500 = model_500.fit(
    train_data_scaled, train_targets,
    epochs=500,
    batch_size=1,
    validation_split=0.1,
    verbose=0
)

mae_history_500 = history_500.history['val_mae']
smooth_mae_history_500 = smooth_curve(mae_history_500[10:])

plt.plot(range(1, len(smooth_mae_history_500) + 1), smooth_mae_history_500)
plt.xlabel('Epochs')
plt.ylabel('Validation MAE')
plt.title('Usrednjeni Validacijski MAE - 500 epoha')
plt.show()

print('Zaključak: Iz grafika se može uočiti tačka gdje model počinje da overfitting-uje ')
print('(kada validacijski MAE prestane opadati i krene blago rasti).')

### j) Evaluacija modela nad testnim skupom podataka

In [ ]:
test_mse_score, test_mae_score = model_500.evaluate(test_data_scaled, test_targets)

print(f'\nTest MSE: {test_mse_score:.4f}')
print(f'Test MAE: {test_mae_score:.4f}')
print(f'Model u prosjeku griješi za: ${test_mae_score * 1000:.2f}')